<a href="https://colab.research.google.com/github/jayanibolisetty14/smartbridge_smartlender/blob/main/Smart_Lender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install flask scikit-learn pandas numpy pyngrok -q

import os
os.makedirs('templates', exist_ok=True)

# --- train_model.py ---
train_code = r'''
import numpy as np
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)
n = 800

gender = np.random.choice(['Male', 'Female'], n, p=[0.8, 0.2])
married = np.random.choice(['Yes', 'No'], n, p=[0.65, 0.35])
dependents = np.random.choice([0, 1, 2, 3], n, p=[0.55, 0.2, 0.15, 0.1])
education = np.random.choice(['Graduate', 'Not Graduate'], n, p=[0.78, 0.22])
self_employed = np.random.choice(['Yes', 'No'], n, p=[0.15, 0.85])
applicant_income = np.random.gamma(5, 1200, n).astype(int) + 1500
coapplicant_income = np.random.choice([0, 1], n, p=[0.4, 0.6]) * (np.random.gamma(3, 800, n).astype(int))
loan_amount = (applicant_income + coapplicant_income) / np.random.uniform(15, 40, n)
loan_amount = loan_amount.astype(int)
loan_term = np.random.choice([120, 180, 240, 300, 360], n, p=[0.05, 0.1, 0.1, 0.15, 0.6])
credit_history = np.random.choice([1, 0], n, p=[0.84, 0.16])
property_area = np.random.choice(['Urban', 'Semiurban', 'Rural'], n, p=[0.38, 0.38, 0.24])

# Rule-based score to generate a realistic target, then add noise
score = (
    credit_history * 4.0
    + (education == 'Graduate') * 0.6
    + (applicant_income + coapplicant_income > 5000) * 1.0
    + (loan_amount < 150) * 0.8
    + (property_area != 'Rural') * 0.5
    - (dependents >= 3) * 0.4
    + np.random.normal(0, 1.1, n)
)
loan_status = (score > 3.2).astype(int)  # 1 = Approved, 0 = Rejected

df = pd.DataFrame({
    'Gender': gender,
    'Married': married,
    'Dependents': dependents,
    'Education': education,
    'Self_Employed': self_employed,
    'ApplicantIncome': applicant_income,
    'CoapplicantIncome': coapplicant_income,
    'LoanAmount': loan_amount,
    'Loan_Amount_Term': loan_term,
    'Credit_History': credit_history,
    'Property_Area': property_area,
    'Loan_Status': loan_status
})

# Encode categoricals (simple manual maps, saved for reuse in Flask app)
maps = {
    'Gender': {'Male': 1, 'Female': 0},
    'Married': {'Yes': 1, 'No': 0},
    'Education': {'Graduate': 1, 'Not Graduate': 0},
    'Self_Employed': {'Yes': 1, 'No': 0},
    'Property_Area': {'Rural': 0, 'Semiurban': 1, 'Urban': 2},
}

X = df.copy()
for col, m in maps.items():
    X[col] = X[col].map(m)

feature_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
                 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
                 'Loan_Amount_Term', 'Credit_History', 'Property_Area']

X_train, X_test, y_train, y_test = train_test_split(
    X[feature_cols], X['Loan_Status'], test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))
print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy: {test_acc:.3f}")

with open('model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'maps': maps, 'features': feature_cols}, f)

print("Saved model.pkl")
'''
with open('train_model.py', 'w') as f:
    f.write(train_code)
get_ipython().system('python train_model.py')

# --- templates ---
home_html = r'''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Smart Lender</title>
<style>
  * { box-sizing: border-box; }
  body { margin:0; font-family: 'Segoe UI', Arial, sans-serif; background:#f4f6fb; color:#1f2937; }
  .navbar { background:#1e3a8a; color:#fff; padding:18px 40px; font-size:22px; font-weight:600; }
  .hero { max-width:900px; margin:60px auto; text-align:center; padding:0 20px; }
  .hero h1 { font-size:38px; margin-bottom:10px; color:#1e3a8a; }
  .hero p { font-size:17px; color:#4b5563; line-height:1.6; }
  .btn { display:inline-block; margin-top:30px; background:#1e3a8a; color:#fff; padding:14px 34px;
         border-radius:8px; text-decoration:none; font-size:16px; font-weight:600; transition:.2s; }
  .btn:hover { background:#152a63; }
  .features { display:flex; gap:20px; max-width:900px; margin:40px auto; padding:0 20px; flex-wrap:wrap; justify-content:center;}
  .card { background:#fff; border-radius:10px; padding:24px; box-shadow:0 2px 8px rgba(0,0,0,.06); flex:1; min-width:220px; }
  .card h3 { margin-top:0; color:#1e3a8a; font-size:18px; }
  .card p { font-size:14px; color:#6b7280; }
</style>
</head>
<body>
  <div class="navbar">💳 Smart Lender</div>
  <div class="hero">
    <h1>Instant Loan Eligibility Prediction</h1>
    <p>Smart Lender uses machine learning to predict loan approval based on applicant details —
       helping banks and financial institutions make faster, data-driven decisions.</p>
    <a class="btn" href="/predict">Check Eligibility →</a>
  </div>
  <div class="features">
    <div class="card"><h3>⚡ Fast</h3><p>Get a prediction in seconds, no manual review needed for low-risk cases.</p></div>
    <div class="card"><h3>🎯 Accurate</h3><p>Trained on applicant income, credit history, and loan details.</p></div>
    <div class="card"><h3>🔒 Reliable</h3><p>Built for credit officers and financial analysts to reduce risk.</p></div>
  </div>
</body>
</html>
'''
with open('templates/home.html', 'w') as f:
    f.write(home_html)

predict_html = r'''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Smart Lender - Apply</title>
<style>
  * { box-sizing: border-box; }
  body { margin:0; font-family:'Segoe UI', Arial, sans-serif; background:#f4f6fb; color:#1f2937; }
  .navbar { background:#1e3a8a; color:#fff; padding:18px 40px; font-size:22px; font-weight:600; }
  .navbar a { color:#cbd5ff; text-decoration:none; float:right; font-size:15px; margin-top:4px; }
  .container { max-width:640px; margin:40px auto; background:#fff; padding:36px; border-radius:12px;
               box-shadow:0 2px 10px rgba(0,0,0,.06); }
  h2 { color:#1e3a8a; margin-top:0; }
  .grid { display:grid; grid-template-columns:1fr 1fr; gap:16px; }
  label { font-size:13px; font-weight:600; color:#374151; display:block; margin-bottom:5px; }
  select, input { width:100%; padding:10px; border:1px solid #d1d5db; border-radius:6px; font-size:14px; }
  .full { grid-column:1 / -1; }
  .btn { margin-top:24px; width:100%; background:#1e3a8a; color:#fff; border:none; padding:14px;
         border-radius:8px; font-size:16px; font-weight:600; cursor:pointer; }
  .btn:hover { background:#152a63; }
</style>
</head>
<body>
  <div class="navbar">💳 Smart Lender <a href="/">Home</a></div>
  <div class="container">
    <h2>Applicant Details</h2>
    <form action="/submit" method="post">
      <div class="grid">
        <div>
          <label>Gender</label>
          <select name="gender" required>
            <option value="Male">Male</option>
            <option value="Female">Female</option>
          </select>
        </div>
        <div>
          <label>Married</label>
          <select name="married" required>
            <option value="Yes">Yes</option>
            <option value="No">No</option>
          </select>
        </div>
        <div>
          <label>Dependents</label>
          <select name="dependents" required>
            <option value="0">0</option>
            <option value="1">1</option>
            <option value="2">2</option>
            <option value="3">3+</option>
          </select>
        </div>
        <div>
          <label>Education</label>
          <select name="education" required>
            <option value="Graduate">Graduate</option>
            <option value="Not Graduate">Not Graduate</option>
          </select>
        </div>
        <div>
          <label>Self Employed</label>
          <select name="self_employed" required>
            <option value="No">No</option>
            <option value="Yes">Yes</option>
          </select>
        </div>
        <div>
          <label>Property Area</label>
          <select name="property_area" required>
            <option value="Urban">Urban</option>
            <option value="Semiurban">Semiurban</option>
            <option value="Rural">Rural</option>
          </select>
        </div>
        <div>
          <label>Applicant Income (₹/month)</label>
          <input type="number" name="applicant_income" min="0" required>
        </div>
        <div>
          <label>Coapplicant Income (₹/month)</label>
          <input type="number" name="coapplicant_income" min="0" value="0">
        </div>
        <div>
          <label>Loan Amount (in thousands)</label>
          <input type="number" name="loan_amount" min="1" required>
        </div>
        <div>
          <label>Loan Term (days)</label>
          <select name="loan_term" required>
            <option value="360">360</option>
            <option value="300">300</option>
            <option value="240">240</option>
            <option value="180">180</option>
            <option value="120">120</option>
          </select>
        </div>
        <div class="full">
          <label>Credit History</label>
          <select name="credit_history" required>
            <option value="1">Has all debts paid (Good)</option>
            <option value="0">Has unpaid debts (Poor)</option>
          </select>
        </div>
      </div>
      <button class="btn" type="submit">Predict Loan Status</button>
    </form>
  </div>
</body>
</html>
'''
with open('templates/predict.html', 'w') as f:
    f.write(predict_html)

submit_html = r'''
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Smart Lender - Result</title>
<style>
  * { box-sizing: border-box; }
  body { margin:0; font-family:'Segoe UI', Arial, sans-serif; background:#f4f6fb; color:#1f2937; }
  .navbar { background:#1e3a8a; color:#fff; padding:18px 40px; font-size:22px; font-weight:600; }
  .navbar a { color:#cbd5ff; text-decoration:none; float:right; font-size:15px; margin-top:4px; }
  .container { max-width:480px; margin:70px auto; background:#fff; padding:40px; border-radius:12px;
               box-shadow:0 2px 10px rgba(0,0,0,.06); text-align:center; }
  .badge { display:inline-block; padding:10px 24px; border-radius:30px; font-size:22px; font-weight:700; margin:20px 0; }
  .approved { background:#dcfce7; color:#15803d; }
  .rejected { background:#fee2e2; color:#b91c1c; }
  p.conf { color:#6b7280; font-size:15px; }
  .btn { display:inline-block; margin-top:20px; background:#1e3a8a; color:#fff; padding:12px 28px;
         border-radius:8px; text-decoration:none; font-weight:600; }
</style>
</head>
<body>
  <div class="navbar">💳 Smart Lender <a href="/">Home</a></div>
  <div class="container">
    <h2>Prediction Result</h2>
    {% if result == "Approved" %}
      <div class="badge approved">✅ Loan Approved</div>
    {% else %}
      <div class="badge rejected">❌ Loan Rejected</div>
    {% endif %}
    <p class="conf">Model confidence: {{ confidence }}%</p>
    <a class="btn" href="/predict">Check Another Applicant</a>
  </div>
</body>
</html>
'''
with open('templates/submit.html', 'w') as f:
    f.write(submit_html)

# --- app.py ---
app_code = r'''
from flask import Flask, render_template, request
import pickle
import numpy as np

app = Flask(__name__)

with open('model.pkl', 'rb') as f:
    data = pickle.load(f)
    model = data['model']
    maps = data['maps']
    features = data['features']


@app.route('/')
def home():
    return render_template('home.html')


@app.route('/predict')
def predict_form():
    return render_template('predict.html')


@app.route('/submit', methods=['POST'])
def submit():
    form = request.form

    gender = maps['Gender'][form.get('gender')]
    married = maps['Married'][form.get('married')]
    dependents = int(form.get('dependents'))
    education = maps['Education'][form.get('education')]
    self_employed = maps['Self_Employed'][form.get('self_employed')]
    applicant_income = float(form.get('applicant_income'))
    coapplicant_income = float(form.get('coapplicant_income') or 0)
    loan_amount = float(form.get('loan_amount'))
    loan_term = float(form.get('loan_term'))
    credit_history = int(form.get('credit_history'))
    property_area = maps['Property_Area'][form.get('property_area')]

    row = np.array([[gender, married, dependents, education, self_employed,
                      applicant_income, coapplicant_income, loan_amount,
                      loan_term, credit_history, property_area]])

    pred = model.predict(row)[0]
    proba = model.predict_proba(row)[0][1]

    result = "Approved" if pred == 1 else "Rejected"
    confidence = round((proba if pred == 1 else 1 - proba) * 100, 1)

    return render_template('submit.html', result=result, confidence=confidence)


if __name__ == '__main__':
    app.run(debug=True)
'''
with open('app.py', 'w') as f:
    f.write(app_code)

# --- run with ngrok ---
NGROK_AUTH_TOKEN = "3FvlvnnujdGOQ4duxwmumRXG9wp_3REzeDD7yt1BLsT3bzzSV"

from pyngrok import ngrok
import threading
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

from app import app as flask_app
public_url = ngrok.connect(5000)
print("Your Smart Lender app is live at:", public_url)
threading.Thread(target=lambda: flask_app.run(port=5000, use_reloader=False)).start()

Train accuracy: 0.984
Test accuracy: 0.994
Saved model.pkl
Your Smart Lender app is live at: NgrokTunnel: "https://wincing-dropout-grouped.ngrok-free.dev" -> "http://localhost:5000"
